In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np
from functools import reduce
import matplotlib.ticker as mtick
import calendar
from datetime import date
import pathlib
from pathlib import Path
from datetime import datetime, timedelta
import os
from matplotlib.ticker import StrMethodFormatter


import seaborn as sns
import matplotlib.pyplot as plt

C:\Users\aallen\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\aallen\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [23]:
#Notes
#Confirm reported electricity use is in kWh before running 
analysis_year = 2025 
timesteps_per_hr = 4
sim_length = 8760 *timesteps_per_hr
steam_htg_value = 970 #btu/lb, initial estimate 
btu_per_kWh = 3412.14 
wh_to_lbs = 0.001 * btu_per_kWh * 1/steam_htg_value


In [25]:
wh_to_lbs*1000

3.5176701030927835

In [3]:
def list_folders_pathlib(directory_path):
    """Lists all immediate subdirectories in the given path."""
    p = pathlib.Path(directory_path)
    return [item.name for item in p.iterdir() if item.is_dir()]

In [62]:
def calc_elec_load_profile_spec(dir_path, id):
    target_string = "default_feature_reports"
    file_name = "default_feature_reports.csv"
    elec_load_x = np.full((sim_length, 1), np.nan)

    # Create the DataFrame using the NumPy array and column names
    elec_load = pd.DataFrame(elec_load_x, columns=['test'])
    elec_sum = 0 
    
    for fol in dir_path.iterdir():
        if (fol.is_dir() and id in fol.name):
            for folder in fol.iterdir():
               if folder.is_dir() and target_string in folder.name:
                new_dir_path = os.path.join(dir_path, fol, folder)
                path_to_file = Path(new_dir_path) / file_name
                loads = pd.read_csv(path_to_file)
                elec_inc = loads['Electricity:Facility(kWh)']
                elec_load[id] = elec_inc 
                elec_sum = elec_load.sum(axis=1).sum()
                elec_load = elec_load.drop(columns=['test'])

    return [elec_load, elec_sum] 

In [26]:
def calc_monthly_sums(year, timesteps_per_hr, load_profile, unit_conv): #for electricity outputs already in kWh, use 1.0 as unit_conv
    start_dt = date(year, 1,1)
    end_dt = date(year+1, 1,1)
    if timesteps_per_hr == 1:
       dates = pd.date_range(start=start_dt, end=end_dt, freq='h')
    elif timesteps_per_hr == 4:
       dates = pd.date_range(start=start_dt, end=end_dt, freq='15min')
    else:
       return "timestep not supported"
 
    dates = dates[:-1] #getting an extra last entry, need to drop
    
    working_df = load_profile.set_index(dates)
    monthly_df = working_df.resample('MS').sum()*unit_conv/(timesteps_per_hr)
    
    
    return working_df, dates, monthly_df

In [34]:
def plot_monthly(id, load_profile, end_use): #Note, output aggregates monthly totals under month start date
    buffer = pd.DateOffset(months=1)  
    if end_use == 'Electricity':
        y_max = load_profile[id].max() * 1.2 
    elif end_use == 'Steam':
        y_max = load_profile['htg'].max() * 1.2 
    ax = plt.gca()
    if end_use == 'Electricity':
        plt.xlim(load_profile[id].index[0] - buffer, load_profile[id].index[-1] + buffer)
        ax.set_xticks(load_profile[id].index[::2])
    elif end_use == 'Steam':
        plt.xlim(load_profile['htg'].index[0] - buffer, load_profile['htg'].index[-1] + buffer)
        ax.set_xticks(load_profile['htg'].index[::2])
    plt.grid()
    plt.ylim(0, y_max)
    plt.xlim()
    if end_use == 'Electricity':
        plt.ylabel('Electricity consumption (kWh)')
        plt.plot(load_profile[id], label=f'Modeled {end_use}')
    elif end_use == 'Steam':
        plt.ylabel('Steam (lbs)')
        plt.plot(load_profile['htg'], label=f'Modeled {end_use}')
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    plt.xticks(rotation=45, ha='right') 
    plt.legend()

In [59]:
def calc_thermal_load_profile_spec(dir_path, id):
    target_string = "default_feature_reports"
    file_name = "default_feature_reports.csv"
    elec_load_x = np.full((sim_length, 1), np.nan)

    # Create the DataFrame using the NumPy array and column names
    thermal_prof = pd.DataFrame(elec_load_x, columns=['test'])
    str1='coil'
    str2='heat'
    str3='reheat'
    str4='cooling coil ice thermal storage'
    str5='cool'
    str6='clg'
    str7='htg'
    
    for fol in dir_path.iterdir():
        if (fol.is_dir() and id in fol.name):
            for folder in fol.iterdir():
               if folder.is_dir() and target_string in folder.name:
                new_dir_path = os.path.join(dir_path, fol, folder)
                path_to_file = Path(new_dir_path) / file_name
                loads = pd.read_csv(path_to_file)
                htg_columns = [col for col in loads.columns if str1 in col.lower() and
                               ((str2 in col.lower()) or (str3 in col.lower()) or (str7 in col.lower())) and (str4 not in col.lower())
                ]
                #for col in htg_columns: #can turn this on for QC
#                     print(col)
#                     print(len(htg_columns))
                clg_columns = [
                col for col in loads.columns
                if str1 in col.lower() and ((str5 in col.lower()) or (str6 in col.lower())) and (str4 not in col.lower())
                 ]
#                 for col in clg_columns: #can turn this on for QC
#                     print(col)
#                     print(len(clg_columns))
                htg_prof = loads[htg_columns].sum(axis=1)
                clg_prof = loads[clg_columns].sum(axis=1)
                thermal_prof['htg'] = htg_prof
                thermal_prof['clg'] = clg_prof 
                thermal_prof = thermal_prof.drop(columns=['test'])

    return thermal_prof

In [67]:
#plot loads and hot water use 
# dir_path_test = Path('C:/Users/aallen/Documents/..../createbar_scenario/') #replace with path 
# id_test = "a20539dd-f643-4532-847a-4fd15663f1bc"
# id_test_2 = '01c99d14-837d-487e-944c-d761eb786a5c'

In [ ]:
#calculate load profiles
# elec_load_01c = calc_elec_load_profile_spec(dir_path_test, id_test_2)[0]
# # elec_sum_01c = calc_elec_load_profile_spec(dir_path_test, id_test_2)[1]

In [ ]:
#create monthly sums 
# month_sums_01c = calc_monthly_sums(analysis_year, timesteps_per_hr, elec_load_01c, 1)[2]

In [ ]:
#plot monthly elec 
# plot_monthly(id_test_2, month_sums_01c, 'Electricity')

In [66]:
#extract thermal loads
# thermal_prof_a20 = calc_thermal_load_profile_spec(dir_path_test, id_test)
# thermal_prof_01c = calc_thermal_load_profile_spec(dir_path_test, id_test_2)

In [ ]:
#create monthly steam sums 
#month_sums_01c_steam = calc_monthly_sums(analysis_year, timesteps_per_hr, thermal_prof_01c['htg'].to_frame(),  wh_to_lbs)[2]
#month_sums_a20_steam_v2 = calc_monthly_sums(analysis_year, timesteps_per_hr, thermal_prof_a20['htg'].to_frame(), wh_to_lbs)[2]

In [ ]:
#plot monthly steam
# plot_monthly(id_test_2, month_sums_01c_steam, 'Steam')